In [1]:
import os
import pandas as pd
import numpy as np

# 1. Đường dẫn file
input_path = os.path.join("..", "data", "processed", "rfm_fcm_clustered.csv")
ab_testing_path = os.path.join("..", "data", "processed", "danh_sach_AB_Testing_chuan.csv")

# 2. Đọc dữ liệu
df = pd.read_csv(input_path)

# 3. Chỉ định CHÍNH XÁC các cột xác suất từ ảnh Excel
prob_cols = ['Prob_Cluster_0', 'Prob_Cluster_1', 'Prob_Cluster_2', 'Prob_Cluster_3']

# 4. ÉP KIỂU BẮT BUỘC toàn bộ các cột này về DẠNG SỐ (FLOAT)
# Giúp sửa triệt để lỗi ô bị dính kiểu Text (dấu tam giác xanh)
for col in prob_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# 5. Trích xuất ma trận U dưới dạng float64
U_matrix = df[prob_cols].to_numpy(dtype=np.float64)

# 6. Tìm Top 1 cluster và xác suất tương ứng
top1_cluster = np.argmax(U_matrix, axis=1)
top1_prob = np.max(U_matrix, axis=1)

# 7. Loại bỏ Top 1 để tìm Top 2
U_copy = U_matrix.copy()
n_samples = len(df)
U_copy[np.arange(n_samples), top1_cluster] = -1.0  # Đặt Top 1 về -1.0

top2_cluster = np.argmax(U_copy, axis=1)
top2_prob = np.max(U_copy, axis=1)

# 8. Tính độ chênh lệch thực tế
diff_prob = top1_prob - top2_prob

# 9. Gán vào DataFrame tạm (KHÔNG GHI ĐÈ FILE GỐC)
df_temp = df.copy()
df_temp["Cum_Top_1"] = top1_cluster
df_temp["Cum_Top_2"] = top2_cluster
df_temp["Xac_Suat_Top_1"] = top1_prob
df_temp["Xac_Suat_Top_2"] = top2_prob
df_temp["Do_Chenh_Lech_Diff"] = diff_prob

# 10. Lọc các khách hàng ranh giới (Diff < 0.15)
boundary_customers = df_temp[df_temp["Do_Chenh_Lech_Diff"] < 0.15].copy()

# Tìm cột ID
id_col = next((col for col in ["customer_unique_id", "customer_id", "id"] if col in df.columns), df.columns[0])

export_cols = [
    id_col,
    "Cum_Top_1",
    "Cum_Top_2",
    "Xac_Suat_Top_1",
    "Xac_Suat_Top_2",
    "Do_Chenh_Lech_Diff"
]

export_df = boundary_customers[export_cols]

# 11. Xuất file CSV danh sách A/B Testing chuẩn
export_df.to_csv(ab_testing_path, index=False)

print(f"✅ Đã xuất thành công file A/B Testing: {ab_testing_path}")
print(f"📊 Tổng số khách hàng thuộc vùng ranh giới (Diff < 0.15): {len(export_df)} dòng")
print("\n10 dòng đầu tiên:")
print(export_df.head(10).to_string(index=False))

✅ Đã xuất thành công file A/B Testing: ..\data\processed\danh_sach_AB_Testing_chuan.csv
📊 Tổng số khách hàng thuộc vùng ranh giới (Diff < 0.15): 17906 dòng

10 dòng đầu tiên:
              customer_unique_id  Cum_Top_1  Cum_Top_2  Xac_Suat_Top_1  Xac_Suat_Top_2  Do_Chenh_Lech_Diff
0000f6ccb0745a6a4b88665a16c9f078          0          3        0.477347        0.332235            0.145112
0006fdc98a402fceb4eb0ee528f6a8d4          3          0        0.453148        0.323623            0.129526
00082cbe03e478190aadbea78542e933          2          3        0.321942        0.270708            0.051234
000949456b182f53c18b68d6babc79c1          0          1        0.472306        0.415219            0.057087
000bfa1d2f1a41876493be685390d6d3          3          0        0.407822        0.402238            0.005584
000e309254ab1fc5ba99dd469d36bdb4          1          0        0.502964        0.372803            0.130161
0011805441c0d1b68b48002f1d005526          2          3        0.399352      

In [2]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Đọc file A/B Testing vừa xuất
ab_path = os.path.join("..", "data", "processed", "danh_sach_AB_Testing_chuan.csv")
df_ab = pd.read_csv(ab_path)

# 2. Tạo cột nhận diện cặp cụm ranh giới (ví dụ: Pair '0-2')
df_ab['Cluster_Pair'] = df_ab['Cum_Top_1'].astype(str) + "_" + df_ab['Cum_Top_2'].astype(str)

# 3. Chia ngẫu nhiên 50/50 phân tầng theo Cluster_Pair
df_control, df_treatment = train_test_split(
    df_ab, 
    test_size=0.5, 
    random_state=42, 
    stratify=df_ab['Cluster_Pair']
)

# 4. Gán nhãn nhóm
df_control['Group'] = 'Control'
df_treatment['Group'] = 'Treatment'

# 5. Gộp lại và xuất file giao cho đội Marketing triển khai
final_ab_df = pd.concat([df_control, df_treatment])
final_path = os.path.join("..", "data", "processed", "danh_sach_AB_Testing_chia_nhom.csv")
final_ab_df.to_csv(final_path, index=False)

print(f"✅ Đã chia nhóm hoàn tất!")
print(f"- Số lượng nhóm Control: {len(df_control)}")
print(f"- Số lượng nhóm Treatment: {len(df_treatment)}")

✅ Đã chia nhóm hoàn tất!
- Số lượng nhóm Control: 8953
- Số lượng nhóm Treatment: 8953


Nhóm Control (Đối chứng - 50%): Áp dụng ưu đãi/kịch bản truyền thông mặc định của Cụm Top 1.

Nhóm Treatment (Thử nghiệm - 50%): Áp dụng ưu đãi/kịch bản truyền thông của Cụm Top 2.